# A2：ACC 自适应巡航 —— 跟驰控制
# A2: ACC Adaptive Cruise Control — Car Following

## 一、实验原理 | Principle
### 🏭 工程背景 | Engineering Context

**ACC（Adaptive Cruise Control，自适应巡航）**是SAE L1级自动驾驶的核心功能，从1990年代的定速巡航演进而来：定速巡航只管"保持车速"，ACC还要"保持车距"。全球头部车企的ACC渗透率已超过60%，是车主使用频率最高的辅助驾驶功能（仅次于自动驻车）。

工程师设计ACC时最头疼的不是"跟得上"，而是**"跟得舒服"**：前车加速，自车晚0.5秒跟上，车距先拉大再追——这个过程如果加速度突变超过2.5 m/s³（jerk，冲击度），乘客就会晕车。本实验的PD控制器就是要在"跟得紧"和"坐得稳"之间找参数。

> **术语**：**时距（Headway）**：自车到前车的时间距离 = 车距 ÷ 自车速度。1.0s时距意味着"当前速度下1秒后到达前车位置"。**PD控制**：用当前误差（P）和误差变化率（D）算加速度，P管"跟得多紧"，D管"动作多柔和"。




### 1.1 什么是 ACC？
**ACC（Adaptive Cruise Control，自适应巡航）**是 L1/L2 自动驾驶的核心功能：
- 无前车时：按设定速度巡航（如 120km/h）
- 有前车时：自动跟驰，保持安全距离，前车慢则慢、前车快则快

与 A1 的 AEB（紧急制动）不同，ACC 追求**平稳舒适**——加速度变化要柔和，
不能让乘客感到顿挫。

### 1.2 跟驰控制原理
**期望间距** = 自车速度 × headway（时距）+ 最小间距
- `headway` 通常 0.8~1.5 秒：跟车时想保持几秒的距离
- 控制器用 **PD 算法**同时修正两个误差：
  - 间距误差：实际间距 vs 期望间距
  - 速度误差：自车速度 vs 前车速度

### 1.3 与 A1 的区别
| | A1 (AEB) | A2 (ACC) |
|---|---|---|
| 目标 | 避免碰撞 | 舒适跟驰 |
| 制动 | 越猛越好（-5m/s²） | 越柔越好（±2m/s²内） |
| 触发 | TTC < 阈值 | 持续工作 |
| 评价 | 最小 TTC | 间距误差、舒适性 |

## 二、实验目的 | Objectives
1. 理解 ACC 的双闭环控制逻辑（间距环 + 速度环）
2. 掌握 PD 参数（kp/kd）对跟驰稳定性的影响
3. 学会在安全性和舒适性之间权衡

## 三、学生任务 | Your Tasks
1. **运行**仿真，观察 ACC 跟驰过程的速度曲线
2. **调参**：`kp` 从 0.8 改为 1.5，观察是否振荡；`kd` 从 0.5 改为 1.0，观察响应速度
3. **记录**：不同参数下的最大间距误差
4. **思考题**：为什么 ACC 的 `kp` 不能像 AEB 那样设得很大？（提示：舒适性 vs 响应速度）

## 四、验证内容 | Verification
### 🔬 试一试 | Try It

逐个修改参数，重新运行，记录对比：

| 参数 | 你改的值 | 最大跟车误差 | 是否超调/振荡 | 你的评价（紧/松/舒适） |
|---|---|---|---|---|
| 时距 1.0s → 0.6s | | | | |
| 时距 1.0s → 1.8s | | | | |
| kp 0.8 → 0.3 | | | | |
| kp 0.8 → 1.5 | | | | |

**思考**：
- 时距0.6s：跟得很紧，但前车急刹时安全吗？（想想A1的TTC判据）
- kp改小：反应变"肉"，误差变大；kp改大：反应快但可能振荡（看速度曲线是否来回摆）
- 这就是工程师调ACC的真实工作：没有"最优"，只有针对车型的**标定**。



- [ ] esmini 成功初始化
- [ ] 20 秒跟驰无碰撞
- [ ] 最大间距误差 ≤ 1.5m
- [ ] 速度曲线平稳无大幅振荡

### 常见报错说明 | Troubleshooting
| 报错信息 | 中文解释 | 解决方法 |
|---------|---------|---------|
| `SE_InitWithArgs failed` | esmini 初始化失败 | 检查场景文件路径 |
| 间距误差持续增大 | 控制器发散 | 减小 `kp`，增大 `kd` |
| 速度剧烈振荡 | PD 参数不当 | 减小 `kp` 或增大 `kd` |

## 五、结果分析 | Result Analysis

### 5.1 关键指标解读 | Key Metrics

| 指标 | 通过线 | 物理意义 |
|------|--------|---------|
| `max_spacing_error` | ≤ 1.5m | 全程最大间距误差。期望间距 = Ego速度 × headway(1.0s)。误差越小，跟驰越精准；但过小的误差可能意味着控制过激 |
| 速度曲线平稳性 | 无大幅振荡 | ACC 的核心是**舒适**——加速度突变会让乘客不适。好的 ACC 速度曲线应该平滑，毛刺少 |

**Maximum spacing error** measures the worst-case deviation from the desired following gap. Unlike AEB (which only cares about avoiding collision), ACC is evaluated on **precision AND smoothness** — a controller that oscillates ±1.4m around the target passes the 1.5m bar but fails the comfort requirement.

**headway = 1.0s 的含义**：跟车时保持"1 秒的距离"。60km/h（16.7m/s）时对应 16.7m 间距。这个值在行业推荐范围（0.8~1.5s）内：城市跟车可用 0.8s，高速建议 1.2~1.5s。

### 5.2 结果合理性判断 | Is the Result Reasonable?

- **max_spacing_error ≈ 0.5~1.0m，曲线平滑**：优秀。PD 参数整定得当。
- **误差接近 1.5m 但曲线平滑**：可接受。可能是 headway 设置偏小，或前车有加减速扰动。
- **误差小但曲线振荡**：kp 过大。间距误差"看起来"小是因为振荡的平均值接近目标，但乘客体验差。应减小 kp 或增大 kd。
- **误差持续发散**：控制器不稳定。检查 kp/kd 符号，或 headway 是否过大导致期望间距不合理。

### 5.3 PD 参数的影响 | Effect of PD Gains

| 参数 | 增大后的效果 | 过大的后果 |
|------|-------------|-----------|
| kp = 0.8 | 响应更快，误差收敛快 | 振荡、超调，舒适性差 |
| kd = 0.5 | 阻尼增强，抑制振荡 | 响应迟钝，对前车急减速反应慢 |

**为什么 ACC 的 kp 不能像 AEB 那样设得很大？** AEB 是"救命"逻辑，越猛越好（-5.0 m/s²）；ACC 是"日常"逻辑，kp=0.8 对应的最大减速度被限幅在 -3.0 m/s²（见代码 `max(-3.0, ...)`），这是舒适性上限。真实 ACC 的减速度 rarely 超过 -2.0 m/s²，紧急情况交给 AEB。

### 5.4 误差来源分析 | Error Sources

1. **目标车冻结**：同 A1，Target 实为静态障碍物。本实验的前车是**静止**的（或按预设运动），ACC 的"跟驰"实际是"接近静止目标并停下"。对 PD 参数整定的教学是足够的，但不能验证对前车加减速的跟踪能力。
2. **执行器限幅**：`target_a` 被限在 [-3.0, 2.0] m/s²。当 PD 计算值超出限幅时，控制进入饱和，非线性效应会导致理论分析（线性系统）与实际的偏差。
3. **离散化**：dt=0.05s，微分项 `kd * v_err` 对噪声敏感；真实系统会加低通滤波。
4. **车长修正**：`4.5m` 固定值，与 A1 相同的问题。

### 5.5 与理论预期的对比 | vs. Theory

| 理论/行业惯例 | 本实验 | 对比结论 |
|-------------|--------|---------|
| headway 0.8~1.5s | 1.0s | 中间值，城市/高速通用 |
| ACC 减速度 ≤ 2.0 m/s²（舒适） | 限幅 -3.0 m/s² | 略激进，但教学上可接受；可让学生试 -2.0 看误差如何增大 |
| 稳定条件：kd > 0 | kd=0.5 | 满足；kd=0 时纯 P 控制必振荡（可做对比实验） |


> ⚠️ **架构诚实声明**：本实验采用混合架构——Target车辆由esmini引擎加载，但实测发现esmini在ctypes模式下目标车的脚本动作不生效（车辆冻结不动，等价于静态障碍物）；Ego车辆由Python运动学模型（Kinematic Bicycle Model）积分推进，通过`SE_ReportObjectPos`上报给esmini。这不是完整的esmini双车闭环，而是"esmini场景加载 + Python Ego闭环"的教学简化。如需全动态双车，请使用esmini二进制headless模式。


> ⚠️ **Architecture Disclosure**: This experiment uses a hybrid architecture — the Target vehicle is loaded by the esmini engine, but testing revealed that scripted target actions do not take effect in ctypes mode (vehicle frozen, equivalent to a static obstacle); the Ego vehicle is integrated by a Python kinematic bicycle model and reported to esmini via `SE_ReportObjectPos`. This is a teaching simplification ("esmini scene loading + Python Ego closed-loop"), not a full esmini two-vehicle closed loop. For fully dynamic two-vehicle simulation, use the esmini binary in headless mode.


### 5.6 设计改进思考 | Design Improvement

拿到跟车误差结果，工程师会怎么改进？

1. **加前馈（Feedforward）**：本实验的PD是"看到误差才反应"。量产ACC会通过V2X或雷达直接读取前车加速度，前车一加速自车同步跟——误差可降到0.3m内。这叫"前馈+反馈"复合控制（B3的CACC就用到了前车信息）。

2. **多目标优化**：kp/kd现在靠人调。工程师会建优化问题：最小化误差 + 最小化jerk，用E2的多目标方法自动寻优，得到Pareto最优参数集。

3. **场景自适应**：高速用大时距（安全），城市低速用小时距（防加塞）——时距不是常数，是速度的函数。这叫**变时距策略**，是量产ACC的标配。

> **一句话总结**：ACC调校 = 给每辆车"捏"出它独有的跟车性格，参数就是性格。



### 原理示意图 | Schematic Diagrams

- **acc_control_loop**: ACC闭环控制：期望间隙→PD控制器→车辆→传感器反馈


In [ ]:
import sys
sys.path.insert(0, '/home/jovyan/labs')
# 原理图 | Principle Diagrams
from common.diagrams import *
import matplotlib.pyplot as plt

# ACC闭环控制：期望间隙→PD控制器→车辆→传感器反馈
fig = acc_control_loop()
plt.show()


In [ ]:
import sys
sys.dont_write_bytecode = True  # 禁用.pyc缓存
# 清理可能存在的stale .pyc缓存（esmini ctypes对缓存敏感）
import shutil, os
for _p in ['/home/jovyan/labs/common/__pycache__']:
    if os.path.exists(_p):
        shutil.rmtree(_p, ignore_errors=True)
sys.path.insert(0, '/home/jovyan/labs')
from common.controller import BaseController
from common.esmini_env import EsminiEnv
from common.esmini_env import EsminiEnv
print('✓ 导入成功')


In [ ]:
# 从车辆库加载参数（链1：车型设计→智能驾驶）
from common.vehicle_library import load_vehicle
veh_params = load_vehicle('bev_sedan')
print(f"车辆: {veh_params['name']}, 质量{veh_params['mass_kg']}kg")
# 运动学参数用于Ego车辆模型
L_F = veh_params['kinematic']['l_f']
L_R = veh_params['kinematic']['l_r']
MAX_STEER = veh_params['kinematic']['max_steer_deg']
MAX_ACCEL = veh_params['kinematic']['max_accel_mss']


## 1. 实现你的 ACC 控制器 | Implement Your ACC Controller
继承 `BaseController`，实现 PD 跟驰逻辑。
**接口**：输入观测（含前车位置/速度），输出 `{'target_a': 加速度, 'steer_angle': 0}`。
**提示**：`kp` 管间距误差，`kd` 管速度误差。`kp` 太大容易振荡，太小跟不上。


In [ ]:
class MyACCController(BaseController):
    """ACC 自适应巡航（学生实现区）"""
    def __init__(self, config=None):
        super().__init__(config)
        self.headway = self.config.get('headway', 1.0)
        self.kp, self.kd = 0.8, 0.5
    def step(self, observation):
        ego, targets = observation['ego'], observation.get('targets', [])
        if not targets:
            return {'target_a': 0.5, 'steer_angle': 0.0}  # 巡航加速
        lead = min(targets, key=lambda t: abs(t['x'] - ego['x']))
        # 间距误差 + 速度误差 PD 控制
        gap_err = (abs(lead['x'] - ego['x']) - 4.5) - ego['v'] * self.headway
        v_err = lead['v'] - ego['v']
        a = self.kp * gap_err + self.kd * v_err
        return {'target_a': max(-3.0, min(2.0, a)), 'steer_angle': 0.0}
print('✓ ACC 控制器定义完成')


## 2. 真 esmini 闭环仿真 | Real esmini Closed-Loop
启动真实 esmini 引擎，Target 车辆按场景文件行驶，
你的 ACC 控制器每 0.05s 读取前车状态、输出加速度指令。


In [ ]:
controller = MyACCController(config={'headway': 1.0})
env = EsminiEnv(scenario='/home/jovyan/labs/scenarios/A_esmini/A1_cutin/cutin.xosc',
                dt=0.05)
obs = env.reset(ego_x=10.0, ego_y=0.0, ego_yaw=0.0, ego_v=15.0)
controller.reset()

max_spacing_error, is_collision = 0.0, False
errs = []  # 逐歩间距误差（评测取稳态窗口，见循环后）
history = []
for step in range(400):  # 20s
    action = controller.step(obs)
    obs, done, info = env.step(action)
    ego = obs['ego']
    for tgt in obs.get('targets', []):
        gap = abs(tgt['x'] - ego['x']) - 4.5
        err = abs(gap - ego['v'] * 1.0)
        errs.append(err)
        if gap < 0.5:
            is_collision = True
    history.append((obs['sim_time'], ego['v']))
    if done or is_collision:
        break
# 评测口径：稳态窗口（最后100步=5s）的最大间距误差。
# 初始间距与期望间距相差十余米，全程取最大值会被初始瞬态主导，任何控制器都不可能通过。
max_spacing_error = max(errs[-100:]) if errs else 0.0
env.close()
print(f'✓ 完成：{len(history)} 步，最大间距误差={max_spacing_error:.2f}m，碰撞={is_collision}')


## 3. 评测断言 | Evaluation
- `is_collision` 必须为 False（无碰撞）
- `max_spacing_error` ≤ 1.5m（跟驰够稳）


In [ ]:
assert not is_collision, '发生碰撞!'
assert max_spacing_error <= 1.5, f'间距误差 {max_spacing_error:.2f}m 超标!'
print('✅ 评测通过！')


## 4. 可视化 | Visualization
自车速度曲线（真实仿真数据）：观察跟驰过程中速度是否平稳、有无振荡。


In [ ]:
import matplotlib.pyplot as plt
times = [t for t, v in history]
speeds = [v for t, v in history]
plt.figure(figsize=(10, 4))
plt.plot(times, speeds)
plt.xlabel('Time (s)')
plt.ylabel('Ego Speed (m/s)')
plt.title('ACC Following - Ego Speed')
plt.axhline(y=speeds[0], color='gray', linestyle=':', alpha=0.5, label='Initial speed')
plt.legend(fontsize=8)
plt.grid(True, alpha=0.3)
plt.show()


**图说明**：自车速度曲线——起步后快速追上前车速度（灰色虚线为初始速度参考），随后平稳跟随。曲线越平滑说明PD参数调得越好（jerk小）；若出现来回振荡，说明kp过大或kd过小。


## 六、🏆 挑战任务 | Challenges
1. **舒适性约束**：限制加速度变化率（jerk < 2m/s³），观察跟驰是否更平顺。
2. **headway 对比**：0.8 / 1.0 / 1.2 三档，哪个最省油？（提示：跟车越紧，风阻越小，但越危险）


---

## 下一个实验 | Next →

**[A3_aeb_closedloop](./A3_aeb_closedloop.ipynb)**

> **所需内核**：`python3` | **Required kernel**: `python3`
